---
title: "Suavizamiento de operaciones de pagos con tarjeta en terminales punto de venta"
date: "2025-01-30"
categories: [series de tiempo, R]
description: "Aquí se realiza un ejercicio de suavizamiento de series de tiempo con información del SIE de Banco De México"
---

El suavizamiento de series de tiempo es una técnica que permite remover de una serie de tiempo elementos aleatorios y estacionales para obtener una tendencia general de la serie. 

Esta técnica puede utilizarse para dar seguimiento a la evolución de la operativa de los pagos con tarjeta de crédito y débito en terminales punto de venta y que son procesados por cámaras de compensación de pagos con tarjeta en México.

En este artículo se muestran diversas técnicas de suavizamiento para el número de operaciones de tarjeta de crédito y débito en terminales punto de venta. Para esto, haremos uso de la API del Sistema de Información Económica (SIE) de Banco de México y haremos uso de librerías de R para aplicar las siguientes técnicas de suavizamiento:

 - Medía móvil
 - Suavizamiento exponencial
 - Seasonal-Trend decomposition using LOESS (STL)
 - Modelod Líneal Dinámico (DLM)

## 1. Obtención de información

Los datos provienen del Sistema de Información Económica (SIE) de Banco de México, que cuenta con una API para descarga programática de información.

Para usar la API es necesario obtener un token de autenticación, el cual puede solicitarse en [esta página web de Banco de México](https://www.banxico.org.mx/SieAPIRest/service/v1/token).

Una forma segura de almacenar el token es mediante variables de entorno, lo que evita exponer credenciales sensibles directamente en el código. Para esto, se crea un archivo `.env` en la carpeta raíz del proyecto donde se almacena el token en formato `BANXICO_TOKEN=tu_token_aqui`. Si bien el token de uso de la API es gratuito y no genera ningún costo, en buena práctica no compartirlo con terceros.

Una vez que el archivo `.env` se encuentra en la carpeta raíz del proyecto, se puede cargar en R usando la librería `dotenv`, que lee el archivo `.env` y hace disponibles sus valores mediante la función `Sys.getenv()`. Por ejemplo, para obtener el token de autenticación se puede usar el siguiente código:

In [30]:
# Cargamos libbrerías y variables de entorno
library(dotenv)
library(here)

load_dot_env(here(".env"))
token <- Sys.getenv("SIE_TOKEN")

Una vez que el token de uso de la API del SIE ha sido cargada, es necesario cargar información. Aquí obtendremos dos series de tiempo, ua de operaciones con tarjeta de débito y otra de operaciones con tarjeta de crédito.

Para esto, haremos uso de las librerías `httr` y `jsonlite` de R.

In [53]:
library(httr)
library(jsonlite)
library(here)

# Id de la serie
serie_id <- "SF351178,SF351179"

url <- paste0("https://www.banxico.org.mx/SieAPIRest/service/v1/series/",serie_id,"/datos")
response <- GET(url, add_headers("Bmx-Token" = token))

# Parsear respuesta
datos_json <- fromJSON(content(response, "text", encoding = "UTF-8"))
df <- datos_json$bmx$series$datos[[1]]

df$dato <- as.numeric(gsub(",", "", df$dato))


# Convertir a numérico, manejando NAs
df$fecha <- as.Date(df$fecha, format = "%d/%m/%Y")

# Contamos número de registros
nrow(df)

# Ordenamos por fecha
df <- df[order(df$fecha),]

#head(df)
head(df)

[1] 1142

,fecha,dato
,<date>,<dbl>
1,2022-12-01,5491357
2,2022-12-02,5868487
3,2022-12-03,5727232
4,2022-12-04,5733744
5,2022-12-05,5666042
6,2022-12-06,5641943


In [50]:
datos_json

$bmx
$bmx$series
   idSerie                                                   titulo
1 SF351179 Número de operaciones realizadas con tarjetas de crédito
2 SF351178  Número de operaciones realizadas con tarjetas de débito
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   